# Sudoku Ultra — RL Bot Training Notebook

Trains a **MaskablePPO** (Stable-Baselines3 + sb3-contrib) bot opponent at three difficulty tiers.

| Tier | Steps | Game delay | Strategy |
|------|-------|------------|----------|
| easy | 50 k | 500 ms–2 s | random cell selection |
| medium | 200 k | 200–500 ms | MRV fallback + PPO |
| hard | 500 k | 100–300 ms | PPO preferred, MRV fallback |

**Architecture**: custom `SudokuEnv` Gymnasium environment with:
- Observation: 81 (board) + 729 (candidate mask) = 810-dimensional
- Action: Discrete(729) — `cell_index * 9 + (digit - 1)`
- Action masking via `get_action_mask()` prevents illegal moves
- Reward: +1 correct, -0.5 wrong, +10 puzzle complete, -0.1/step

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import sys, pathlib
# Add the ml-service root to sys.path so app.* imports work.
ROOT = pathlib.Path("../../services/ml-service").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")

print("Python:", sys.version)
print("Working dir:", pathlib.Path.cwd())

In [ ]:
# ── Verify dependencies ───────────────────────────────────────────────────────
import stable_baselines3, sb3_contrib, gymnasium
print("stable-baselines3:", stable_baselines3.__version__)
print("sb3-contrib:", sb3_contrib.__version__)
print("gymnasium:", gymnasium.__version__)

In [ ]:
# ── Generate puzzle pool and inspect env ─────────────────────────────────────
from app.ml.sudoku_env import SudokuEnv, generate_puzzle_pool

pool = generate_puzzle_pool(n=200, clues=32)
print(f"Pool size: {len(pool)} puzzles")

env = SudokuEnv(pool, max_steps=400, render_mode="ansi")
obs, _ = env.reset()

print(f"\nObservation shape: {obs.shape}")
print(f"Action space: {env.action_space}")
print(f"\nInitial board:")
print(env.render())

In [ ]:
# ── Verify action masking ─────────────────────────────────────────────────────
import numpy as np

mask = env.get_action_mask()
valid_actions = np.where(mask)[0]
print(f"Valid actions: {len(valid_actions)} / 729")
print(f"Empty cells: {np.sum(obs[:81] == 0)}")

# Check: valid_actions should only target empty cells.
for a in valid_actions:
    cell = a // 9
    assert obs[cell] == 0, f"Action {a} targets non-empty cell {cell}"
print("✓ All valid actions target empty cells")

In [ ]:
# ── Train EASY tier (50k steps — fast, ~2 min on CPU) ─────────────────────────
from app.ml.train_rl_bot import train_and_save

easy_metrics = train_and_save("easy")
print(f"\nEasy bot results:")
for k, v in easy_metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

In [ ]:
# ── Train MEDIUM tier (200k steps — ~8 min on CPU) ────────────────────────────
medium_metrics = train_and_save("medium")
print(f"\nMedium bot results:")
for k, v in medium_metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

In [ ]:
# ── Train HARD tier (500k steps — ~20 min on CPU, or GPU recommended) ────────
hard_metrics = train_and_save("hard")
print(f"\nHard bot results:")
for k, v in hard_metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

In [ ]:
# ── Compare all tiers ─────────────────────────────────────────────────────────
import pandas as pd

all_metrics = [
    {"tier": m["tier"], "win_rate": m["win_rate"], "mean_reward": m["mean_reward"],
     "steps_to_win_mean": m["steps_to_win_mean"]}
    for m in [easy_metrics, medium_metrics, hard_metrics]
]
df = pd.DataFrame(all_metrics).set_index("tier")
df["win_rate"] = df["win_rate"].map("{:.1%}".format)
df["mean_reward"] = df["mean_reward"].round(3)
df["steps_to_win_mean"] = df["steps_to_win_mean"].round(1)
print(df.to_string())

In [ ]:
# ── Verify bot_service get_move works for all tiers ──────────────────────────
from app.services.bot_service import bot_service
from app.ml.sudoku_env import generate_puzzle

board, solution = generate_puzzle(clues=32)

for tier in ("easy", "medium", "hard"):
    result = bot_service.get_move(board[:], solution, tier=tier)
    print(f"[{tier}] cell={result['cell_index']:2d} digit={result['digit']} "
          f"confidence={result['confidence']:.3f} source={result['source']}")
    assert solution[result["cell_index"]] == result["digit"], "Bot chose wrong digit!"
print("✓ All tiers return correct moves")

In [ ]:
# ── Inspect MLflow runs ───────────────────────────────────────────────────────
import mlflow

client = mlflow.MlflowClient()
runs = client.search_runs(
    experiment_ids=["0"],
    filter_string="tags.mlflow.runName LIKE 'rl_bot_%'",
    order_by=["start_time DESC"],
    max_results=3,
)

for run in runs:
    print(f"Run: {run.info.run_name} | status={run.info.status}")
    for k, v in run.data.metrics.items():
        print(f"  {k}: {v:.4f}")

In [ ]:
# ── Saved model files ─────────────────────────────────────────────────────────
import pathlib

models_dir = pathlib.Path("../../services/ml-service/ml/models")
for f in sorted(models_dir.glob("rl_bot_*.zip")):
    size_kb = f.stat().st_size // 1024
    print(f"  {f.name}  ({size_kb} KB)")